[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/rl-lab/blob/main/04_q_learning.ipynb)


# 04. Табличный Q-learning на FrozenLake

Цель ноутбука — обучить модельно-свободного агента на той же среде FrozenLake-v1 и сравнить его с Value Iteration.

**Результаты обучения:**
- применять epsilon-greedy стратегию исследования;
- обновлять Q-таблицу по TD-ошибке;
- оценивать жадную политику после обучения;
- сопоставлять модельный и модельно-свободный подходы.

## Источник
Lapan M., *Deep Reinforcement Learning Hands-On*, глава 5; Sutton R. S., Barto A. G., глава 6.


In [ ]:
!pip install -q gymnasium


Импортируем библиотеки и зададим seed. Среда будет такой же, как в ноутбуке 03: `FrozenLake-v1` со скользкими переходами.


In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
except ImportError:
    pass


Создаем среду и фиксируем ее размерности. Q-learning не читает модель `P`, но размер таблицы нужен из пространств состояний и действий.


In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=True)
obs, info = env.reset(seed=SEED)
nS = env.observation_space.n
nA = env.action_space.n
print("states:", nS, "actions:", nA)


Задаем гиперпараметры и Q-таблицу. Epsilon линейно убывает от 1.0 до 0.05 за первые 5000 эпизодов.


In [ ]:
ALPHA = 0.1
GAMMA = 0.99
EPS_START, EPS_END = 1.0, 0.05
EPS_DECAY_EPISODES = 5000
N_EPISODES = 20000
Q = np.zeros((nS, nA))


В цикле обучения агент действует epsilon-greedy и обновляет выбранное значение Q по однокроковому TD-таргету.


In [ ]:
episode_rewards = []
total_steps = 0
import time
t0 = time.perf_counter()

for episode in range(N_EPISODES):
    frac = episode / EPS_DECAY_EPISODES
    eps = max(EPS_END, EPS_START - frac * (EPS_START - EPS_END))
    obs, info = env.reset(seed=SEED + episode)
    done, total_reward = False, 0.0
    while not done:
        if np.random.random() < eps:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(Q[obs]))
        next_obs, reward, terminated, truncated, info = env.step(action)
        target = reward + GAMMA * Q[next_obs].max() * (not terminated)
        Q[obs, action] += ALPHA * (target - Q[obs, action])
        obs = next_obs
        total_reward += reward
        total_steps += 1
        done = terminated or truncated
    episode_rewards.append(total_reward)
elapsed_ql = time.perf_counter() - t0


Скользящее среднее по 100 эпизодам сглаживает редкие успехи и показывает динамику обучения.


In [ ]:
rolling = pd.Series(episode_rewards).rolling(100).mean()
plt.figure(figsize=(8, 4))
plt.plot(rolling)
plt.xlabel("Episode")
plt.ylabel("Rolling mean reward")
plt.title("Q-learning on FrozenLake-v1")
plt.grid(alpha=0.3)
plt.show()


После обучения оцениваем жадную политику без исследования. Эта оценка напрямую сопоставима с тестом политики из ноутбука 03.


In [ ]:
ql_policy = np.argmax(Q, axis=1)
test_rewards = []
for ep in range(1000):
    obs, info = env.reset(seed=SEED + ep)
    done, total_reward = False, 0.0
    while not done:
        obs, reward, terminated, truncated, info = env.step(int(ql_policy[obs]))
        total_reward += reward
        done = terminated or truncated
    test_rewards.append(total_reward)

ql_success_rate = np.mean(np.array(test_rewards) == 1.0)
print("q_learning_success_rate:", ql_success_rate)


Для сравнения попробуем прочитать summary из ноутбука 03. Если его еще не запускали, таблица честно оставит недоступные значения как `-`.


In [ ]:
vi_elapsed = vi_success = match_pct = "-"
if os.path.exists("results/03_value_iteration_summary.csv"):
    vi_summary = pd.read_csv("results/03_value_iteration_summary.csv")
    vi_map = dict(zip(vi_summary["metric"], vi_summary["value"]))
    vi_elapsed = vi_map.get("elapsed_sec", "-")
    vi_success = vi_map.get("success_rate", "-")
    vi_env = gym.make("FrozenLake-v1", is_slippery=True)
    P = vi_env.unwrapped.P
    V = np.zeros(nS)
    for _ in range(10000):
        V_new = np.array([max(sum(p * (r + GAMMA * V[ns] * (not term))
            for p, ns, r, term in P[s][a]) for a in range(nA)) for s in range(nS)])
        if np.max(np.abs(V_new - V)) < 1e-8: break
        V = V_new
    vi_policy = np.array([np.argmax([sum(p * (r + GAMMA * V[ns] * (not term))
        for p, ns, r, term in P[s][a]) for a in range(nA)]) for s in range(nS)])
    match_pct = f"{100 * np.mean(ql_policy == vi_policy):.1f}%"
else:
    print("Запустите notebook 03, чтобы заполнить VI-колонки сравнения.")


Based on Lapan M., *Deep Reinforcement Learning Hands-On*, chapter 5.


In [ ]:
os.makedirs("results", exist_ok=True)
df_results = pd.DataFrame([
    ("requires_model", "yes", "no"),
    ("interaction_samples", 0, total_steps),
    ("training_time_sec", vi_elapsed, elapsed_ql),
    ("test_success_rate", vi_success, ql_success_rate),
    ("policy_match_optimal", "100%", match_pct),
], columns=["metric", "value_iteration", "q_learning"])
print(df_results.to_string(index=False))
df_results.to_csv("results/04_q_learning.csv", index=False)
